# Cell 1: Installation and Imports
This step installs the necessary libraries and establishes a BigQuery Client connection.

In [20]:
import pandas as pd
from google.cloud import bigquery
from datetime import datetime
import math
import os
from dotenv import load_dotenv, find_dotenv

# Load environment variables
load_dotenv(find_dotenv())

PROJECT_ID = os.getenv('PROJECT_ID')
client = bigquery.Client(project=PROJECT_ID)

print(f"[{datetime.now()}] BigQuery Client created. Project ID: {PROJECT_ID}")

[2026-06-08 16:00:00.049474] BigQuery Client created. Project ID: my-project-for-bigquery-445809


## Connection Test
Verify that the BigQuery client can successfully connect and list datasets to ensure authentication is working correctly.

In [21]:
print(f"[{datetime.now()}] Testing connection...")
try:
    datasets = list(client.list_datasets(max_results=5))
    print(f"Connection successful! Found {len(datasets)} datasets (showing up to 5):")
    for ds in datasets:
        print(f" - Dataset: {ds.dataset_id}")
except Exception as e:
    print(f"Connection failed: {e}")

[2026-06-08 16:00:04.709132] Testing connection...
Connection successful! Found 5 datasets (showing up to 5):
 - Dataset: client_example_81cd1366_d99b_4064_a611_4d74d9e91c1c
 - Dataset: dbt_ecommerce_intermediate
 - Dataset: dbt_ecommerce_marts
 - Dataset: dbt_ecommerce_staging
 - Dataset: raw_ecommerce


## Step 1: Model Training
Train an LTV prediction model using `BOOSTED_TREE_REGRESSOR`. The target value is `actual_profit`, and features include `recency`, `frequency`, `total_views`, `total_cart_adds`, and `compound_score`.

In [8]:
print(f"[{datetime.now()}] Step 1: Starting model training...")

train_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.dbt_ecommerce_intermediate.int_ltv_model`
OPTIONS(
    model_type = 'BOOSTED_TREE_REGRESSOR',
    input_label_cols = ['actual_profit'],
    max_iterations = 50,
    subsample = 0.8,
    data_split_method = 'NO_SPLIT'
) AS
SELECT
    recency,
    frequency,
    total_views,
    total_cart_adds,
    compound_score,
    actual_profit
FROM `{PROJECT_ID}.dbt_ecommerce_intermediate.int_ltv_training_features`
WHERE actual_profit IS NOT NULL
"""

try:
    query_job = client.query(train_sql)
    query_job.result()
    print(f"[{datetime.now()}] Step 1: Model training complete.")
except Exception as e:
    print(f"[{datetime.now()}] Step 1 Error: {e}")

[2026-06-08 11:41:57.148166] Step 1: Starting model training...
[2026-06-08 12:01:35.222725] Step 1: Model training complete.


## Step 2: Model Evaluation
Query model evaluation metrics (MAE, MSE, R2) and calculate RMSE for monitoring.

In [9]:
print(f"[{datetime.now()}] Step 2: Starting model evaluation...")

eval_sql = f"""
SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.dbt_ecommerce_intermediate.int_ltv_model`)
"""

try:
    eval_results = client.query(eval_sql).to_dataframe()
    mae = eval_results['mean_absolute_error'][0]
    mse = eval_results['mean_squared_error'][0]
    r2 = eval_results['r2_score'][0]
    rmse = math.sqrt(mse)
    
    print(f"MAE: {mae}")
    print(f"MSE: {mse}")
    print(f"R2 Score: {r2}")
    print(f"RMSE: {rmse}")
    
    display(eval_results)
    print(f"[{datetime.now()}] Step 2: Evaluation complete.")
except Exception as e:
    print(f"[{datetime.now()}] Step 2 Error: {e}")

[2026-06-08 12:04:20.811496] Step 2: Starting model evaluation...


/Volumes/Transcend/Profile/marketing/ecommerce_dataset/venv/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


MAE: 47.95915229988145
MSE: 12253.574964550058
R2 Score: 0.12707766673120546
RMSE: 110.69586697140078


,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,47.959152,12253.574965,7.897691,23.557494,0.127078,0.129835


[2026-06-08 12:04:22.278120] Step 2: Evaluation complete.


## Step 3: Inference
Perform LTV predictions for all users and store the results in the `int_ltv_predictions` table.

In [22]:
print(f"[{datetime.now()}] Step 3: Starting inference...")

predict_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.dbt_ecommerce_intermediate.int_ltv_predictions` AS
SELECT
    user_id,
    predicted_actual_profit AS predicted_profit_90_days,
    (SELECT feature FROM UNNEST(top_feature_attributions) ORDER BY attribution DESC LIMIT 1) AS primary_driver,
    (SELECT feature FROM UNNEST(top_feature_attributions) ORDER BY attribution ASC LIMIT 1) AS primary_barrier
FROM ML.EXPLAIN_PREDICT(MODEL `{PROJECT_ID}.dbt_ecommerce_intermediate.int_ltv_model`,
    (
        SELECT
            rfm.user_id,
            rfm.recency,
            rfm.frequency,
            COALESCE(evt.total_views, 0) AS total_views,
            COALESCE(evt.total_cart_adds, 0) AS total_cart_adds,
            COALESCE(sent.compound_score, 0) AS compound_score
        FROM `{PROJECT_ID}.dbt_ecommerce_intermediate.int_rfm_scores` rfm
        LEFT JOIN `{PROJECT_ID}.dbt_ecommerce_intermediate.int_event_aggregates` evt ON rfm.user_id = evt.user_id
        LEFT JOIN `{PROJECT_ID}.dbt_ecommerce_intermediate.int_review_sentiment_user` sent ON rfm.user_id = sent.user_id
    ),
    STRUCT(3 AS top_k_features)
)
"""

try:
    client.query(predict_sql).result()
    print(f"[{datetime.now()}] Step 3: Inference complete and results stored in the table.")
except Exception as e:
    print(f"[{datetime.now()}] Step 3 Error: {e}")

[2026-06-08 16:00:56.428603] Step 3: Starting inference...
[2026-06-08 16:00:59.043039] Step 3: Inference complete and results stored in the table.


## Step 4: Feature Importance
Analyze feature importance and store the results in `mart_model_explanations`.

In [23]:
print(f"[{datetime.now()}] Step 4: Starting feature importance analysis...")

importance_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.dbt_ecommerce_marts.mart_model_explanations` AS
SELECT * FROM ML.FEATURE_IMPORTANCE(MODEL `{PROJECT_ID}.dbt_ecommerce_intermediate.int_ltv_model`)
"""

try:
    client.query(importance_sql).result()
    # To display, we query it back
    display_df = client.query(f"SELECT * FROM `{PROJECT_ID}.dbt_ecommerce_marts.mart_model_explanations` ORDER BY importance_weight DESC").to_dataframe()
    display(display_df)
    print(f"[{datetime.now()}] Step 4: Feature importance analysis complete.")
except Exception as e:
    print(f"[{datetime.now()}] Step 4 Error: {e}")

[2026-06-08 16:01:03.695059] Step 4: Starting feature importance analysis...


/Volumes/Transcend/Profile/marketing/ecommerce_dataset/venv/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,feature,importance_weight,importance_gain,importance_cover
0,recency,110,65754.282246,605.309091
1,compound_score,78,153335.889233,1155.435897
2,total_views,48,75148.065890,319.583333
3,frequency,34,116901.809399,599.058824
4,total_cart_adds,26,52021.556004,172.000000


[2026-06-08 16:01:06.407838] Step 4: Feature importance analysis complete.


## Step 5: Model Monitoring
Query training history and log the execution metrics to `model_run_log`. If RMSE increases by more than 15% compared to the previous run, `retrain_triggered` will be set to True.

In [12]:
print(f"[{datetime.now()}] Step 5: Starting model monitoring and logging...")

try:
    # 1. Query training history
    train_info_sql = f"SELECT * FROM ML.TRAINING_INFO(MODEL `{PROJECT_ID}.dbt_ecommerce_intermediate.int_ltv_model`)"
    train_info_df = client.query(train_info_sql).to_dataframe()
    print("Training History Info:")
    display(train_info_df)

    # 2. Check if log table exists, create if not
    log_table_id = f"{PROJECT_ID}.dbt_ecommerce_intermediate.model_run_log"
    create_log_sql = f"""
    CREATE TABLE IF NOT EXISTS `{log_table_id}` (
        run_id STRING,
        run_timestamp TIMESTAMP,
        rmse FLOAT64,
        mae FLOAT64,
        retrain_triggered BOOL
    )
    """
    client.query(create_log_sql).result()

    # 3. Get previous RMSE to determine if retraining is triggered
    last_run_sql = f"SELECT rmse FROM `{log_table_id}` ORDER BY run_timestamp DESC LIMIT 1"
    last_run_df = client.query(last_run_sql).to_dataframe()
    
    retrain_triggered = False
    if not last_run_df.empty:
        last_rmse = last_run_df['rmse'][0]
        if rmse > last_rmse * 1.15:
            retrain_triggered = True
            print(f"ALERT: Current RMSE ({rmse:.4f}) is more than 15% higher than previous ({last_rmse:.4f})!")

    # 4. Write current run record
    run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
    insert_sql = f"""
    INSERT INTO `{log_table_id}` (run_id, run_timestamp, rmse, mae, retrain_triggered)
    VALUES ('{run_id}', CURRENT_TIMESTAMP(), {rmse}, {mae}, {retrain_triggered})
    """
    client.query(insert_sql).result()
    
    print(f"[{datetime.now()}] Step 5: Execution log updated to {log_table_id}.")
    
    # Display last 5 records
    log_display_df = client.query(f"SELECT * FROM `{log_table_id}` ORDER BY run_timestamp DESC LIMIT 5").to_dataframe()
    display(log_display_df)

except Exception as e:
    print(f"[{datetime.now()}] Step 5 Error: {e}")

[2026-06-08 12:08:53.716108] Step 5: Starting model monitoring and logging...
Training History Info:


/Volumes/Transcend/Profile/marketing/ecommerce_dataset/venv/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,training_run,iteration,loss,eval_loss,learning_rate,duration_ms
0,0,5,110.696,110.696,0.3,38
1,0,4,111.707,111.707,0.3,56
2,0,3,113.097,113.097,0.3,51
3,0,2,115.014,115.014,0.3,49
4,0,1,118.415,118.415,0.3,951661


/Volumes/Transcend/Profile/marketing/ecommerce_dataset/venv/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


[2026-06-08 12:08:56.723697] Step 5: Execution log updated to my-project-for-bigquery-445809.dbt_ecommerce_intermediate.model_run_log.


,run_id,run_timestamp,rmse,mae,retrain_triggered
0,20260608_120855,2026-06-08 04:08:55.539195+00:00,110.695867,47.959152,False
